# Pipeline 2 — Single-Stage RAG (Dense / FAISS Retrieval)

**Method:** Sentence-transformer embeddings → FAISS cosine similarity → top-k → LLM.

```
Question ──→ Dense Embed ──→ FAISS Search (top-k) ──→ LLM Generate Answer
```

**Features:**
- 14-key Groq API carousel with rate-limit tracking
- Parallel sample processing (14 workers)
- Dense model: `all-MiniLM-L6-v2` (384-dim)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q datasets numpy sentence-transformers faiss-cpu groq tqdm

## Step 1b — Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Path to your project folder on Google Drive ---
BASE_DIR     = '/content/drive/MyDrive/HotPotQA-Coding-Trials'
API_KEYS_CSV = f'{BASE_DIR}/api_keys.csv'
RESULTS_DIR  = f'{BASE_DIR}/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Drive mounted. Project dir: {BASE_DIR}")

## Step 2 — Imports and Configuration

In [ ]:
import os, json, re, time, random, collections, string, csv, threading
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq
from datetime import datetime

GROQ_MODEL    = "llama-3.3-70b-versatile"
DENSE_MODEL   = "all-MiniLM-L6-v2"
TOP_K         = 4
N_SAMPLES     = 50
SEED          = 42
MAX_WORKERS   = 14
PIPELINE_NAME = "SingleStageRAG"
# API_KEYS_CSV and RESULTS_DIR are set in the Drive mount cell above

print(f"Pipeline : {PIPELINE_NAME}")
print(f"LLM      : {GROQ_MODEL} | Dense: {DENSE_MODEL}")
print(f"Workers  : {MAX_WORKERS} | Top-K: {TOP_K} | Samples: {N_SAMPLES}")

## Step 3 — API Key Manager (14-Key Carousel)

In [ ]:
class APIKeyManager:
    BUFFERS = {'rpm': 25, 'rpd': 900, 'tpm': 10_000, 'tpd': 90_000}

    def __init__(self, csv_path):
        self.keys = self._load_keys(csv_path)
        self._lock = threading.Lock()
        self._idx = 0
        self._usage = {k: {'min_req': [], 'day_req': [], 'min_tok': [], 'day_tok': []} for k in self.keys}
        print(f"Loaded {len(self.keys)} API keys")

    @staticmethod
    def _load_keys(csv_path):
        keys = []
        with open(csv_path, 'r') as f:
            for row in csv.DictReader(f):
                k = row.get('API_KEY', '').strip()
                if k: keys.append(k)
        if not keys: raise ValueError(f"No keys in {csv_path}")
        return keys

    def _clean(self, key):
        now = time.time()
        u = self._usage[key]
        u['min_req'] = [t for t in u['min_req'] if now - t < 60]
        u['day_req'] = [t for t in u['day_req'] if now - t < 86400]
        u['min_tok'] = [(t, n) for t, n in u['min_tok'] if now - t < 60]
        u['day_tok'] = [(t, n) for t, n in u['day_tok'] if now - t < 86400]

    def _is_available(self, key):
        self._clean(key)
        u = self._usage[key]
        return (len(u['min_req']) < self.BUFFERS['rpm']
                and len(u['day_req']) < self.BUFFERS['rpd']
                and sum(n for _, n in u['min_tok']) < self.BUFFERS['tpm']
                and sum(n for _, n in u['day_tok']) < self.BUFFERS['tpd'])

    def get_key(self):
        with self._lock:
            for _ in range(len(self.keys)):
                key = self.keys[self._idx]
                self._idx = (self._idx + 1) % len(self.keys)
                if self._is_available(key): return key
            return self._wait_and_get()

    def _wait_and_get(self):
        min_wait = 60
        for key in self.keys:
            reqs = self._usage[key]['min_req']
            if reqs: min_wait = min(min_wait, max(0, 60 - (time.time() - min(reqs))))
        print(f"  ⏳ All keys busy — waiting {min_wait:.1f}s...")
        time.sleep(min_wait + 1)
        for _ in range(len(self.keys)):
            key = self.keys[self._idx]
            if self._is_available(key): return key
            self._idx = (self._idx + 1) % len(self.keys)
        return self.keys[self._idx]

    def record(self, key, tokens=0):
        with self._lock:
            now = time.time()
            self._usage[key]['min_req'].append(now)
            self._usage[key]['day_req'].append(now)
            if tokens > 0:
                self._usage[key]['min_tok'].append((now, tokens))
                self._usage[key]['day_tok'].append((now, tokens))

    def mark_exhausted(self, key):
        with self._lock:
            self._usage[key]['min_req'].extend([time.time()] * self.BUFFERS['rpm'])
            self._idx = (self._idx + 1) % len(self.keys)

    def status(self):
        with self._lock:
            for i, key in enumerate(self.keys):
                self._clean(key)
                u = self._usage[key]
                print(f"  Key {i+1:2d}: {len(u['min_req']):3d}/{self.BUFFERS['rpm']} RPM  "
                      f"{len(u['day_req']):4d}/{self.BUFFERS['rpd']} RPD")

key_manager = APIKeyManager(API_KEYS_CSV)

## Step 4 — Load HotPotQA Data

In [ ]:
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
if N_SAMPLES is not None:
    random.seed(SEED)
    samples = ds.select(random.sample(range(len(ds)), min(N_SAMPLES, len(ds))))
else:
    samples = ds
print(f"Loaded {len(samples)} samples")

## Step 5 — Data Processing Utilities

In [ ]:
def process_context(context):
    candidates = []
    for title, sentences in zip(context['title'], context['sentences']):
        for i, sent in enumerate(sentences):
            candidates.append({'title': title, 'sent_id': i, 'text': sent})
    return candidates

def format_gold_supporting_facts(supporting_facts):
    return [{'title': t, 'sent_id': s}
            for t, s in zip(supporting_facts['title'], supporting_facts['sent_id'])]

## Step 6 — Dense Retriever (FAISS)

In [ ]:
embed_model = SentenceTransformer(DENSE_MODEL)
print(f"Dense model loaded: {DENSE_MODEL}")

# Thread lock for FAISS (not thread-safe for index building)
_faiss_lock = threading.Lock()

def dense_retrieve(query, candidates, k=5):
    if not candidates: return []
    texts = [c['text'] for c in candidates]
    cand_emb = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    q_emb = embed_model.encode([query], convert_to_numpy=True, show_progress_bar=False)[0]
    faiss.normalize_L2(cand_emb)
    faiss.normalize_L2(q_emb.reshape(1, -1))
    with _faiss_lock:
        index = faiss.IndexFlatL2(cand_emb.shape[1])
        index.add(cand_emb)
        distances, indices = index.search(q_emb.reshape(1, -1), min(k, len(candidates)))
    return [dict(**candidates[int(idx)], score=float(1.0 / (1.0 + dist)))
            for dist, idx in zip(distances[0], indices[0])]

## Step 7 — LLM Client (Multi-Key Groq)

In [ ]:
class GroqClient:
    def __init__(self, model_name, km):
        self.model_name = model_name
        self.km = km
        self._clients = {}
        self._clock = threading.Lock()

    def _client_for(self, api_key):
        with self._clock:
            if api_key not in self._clients:
                self._clients[api_key] = Groq(api_key=api_key)
            return self._clients[api_key]

    def generate(self, prompt, system_prompt=None, max_retries=3):
        for attempt in range(max_retries):
            api_key = self.km.get_key()
            client = self._client_for(api_key)
            try:
                msgs = []
                if system_prompt: msgs.append({"role": "system", "content": system_prompt})
                msgs.append({"role": "user", "content": prompt})
                resp = client.chat.completions.create(
                    model=self.model_name, messages=msgs, max_tokens=512, temperature=0.1)
                tokens = resp.usage.total_tokens if resp.usage else 0
                self.km.record(api_key, tokens)
                return resp.choices[0].message.content
            except Exception as e:
                if '429' in str(e) or 'rate_limit' in str(e).lower():
                    self.km.mark_exhausted(api_key); continue
                print(f"  LLM error: {e}"); return ""
        return ""

    @staticmethod
    def parse_json_output(text):
        try: return json.loads(text)
        except json.JSONDecodeError: pass
        m = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except: pass
        m = re.search(r"(\{.*\})", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(1))
            except: pass
        return {"answer": "JSON_PARSE_ERROR", "supporting_facts": [], "raw_output": text}

    def predict(self, prompt):
        return self.parse_json_output(self.generate(prompt))

llm = GroqClient(GROQ_MODEL, key_manager)
print("LLM client ready (multi-key).")

## Step 8 — Prompt Construction

In [ ]:
def construct_prompt(question, retrieved_sentences):
    context_str = ""
    for i, item in enumerate(retrieved_sentences, 1):
        context_str += (f"[{i}] Title: {item['title']}\n"
                        f"    Sentence ID: {item['sent_id']}\n"
                        f"    Text: {item['text']}\n\n")
    return f"""You are a helpful assistant for Question Answering.
Answer the following question based ONLY on the provided context sentences.
You must also identify which sentences support your answer.

Context:
{context_str}

Question: {question}

Instructions:
1. Provide a short, concise answer.
2. List the supporting facts as title + sent_id pairs.
3. Use EXACT titles and sent_ids from the context.
4. Output valid JSON only.

Format:
{{{{
  "answer": "...",
  "supporting_facts": [{{{{"title": "...", "sent_id": N}}}}, ...]
}}}}"""

## Step 9 — Evaluator

In [ ]:
def normalize_answer(s):
    s = str(s) if s is not None else ""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    return ' '.join(s.split())

def answer_f1(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = collections.Counter(pt) & collections.Counter(gt)
    ns = sum(common.values())
    if ns == 0: return 0.0
    p, r = ns / len(pt), ns / len(gt)
    return (2 * p * r) / (p + r)

def answer_em(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def sp_metrics(pred_sp, gold_sp):
    def to_set(lst):
        return {(x['title'], x['sent_id']) if isinstance(x, dict) else (x[0], x[1]) for x in lst}
    ps, gs = to_set(pred_sp), to_set(gold_sp)
    tp = len(ps & gs)
    prec = tp / len(ps) if ps else 0.0
    rec  = tp / len(gs) if gs else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    em   = 1.0 if ps == gs and len(gs) > 0 else 0.0
    return {'sp_em': em, 'sp_f1': f1, 'sp_prec': prec, 'sp_recall': rec}

## Step 10 — Run Single-Stage Dense RAG (Parallel)

In [ ]:
def process_sample(sample):
    t0 = time.time()
    question   = sample['question']
    candidates = process_context(sample['context'])
    retrieved  = dense_retrieve(question, candidates, k=TOP_K)
    prompt     = construct_prompt(question, retrieved)
    resp       = llm.predict(prompt)
    elapsed    = time.time() - t0
    gold_sp    = format_gold_supporting_facts(sample['supporting_facts'])
    return {
        'pred': {'answer': resp.get('answer', ''), 'supporting_facts': resp.get('supporting_facts', [])},
        'gold': {'answer': sample['answer'], 'supporting_facts': gold_sp},
        'detail': {
            'id': sample['id'], 'question': question,
            'gold_answer': sample['answer'], 'gold_sp': gold_sp,
            'pred_answer': resp.get('answer', ''), 'pred_sp': resp.get('supporting_facts', []),
            'retrieved_context': retrieved, 'time_taken': elapsed,
            'pipeline': PIPELINE_NAME, 'raw_prediction': resp,
        }
    }

experiment_start = datetime.now()
print(f"Starting {PIPELINE_NAME} at {experiment_start.strftime('%H:%M:%S')}")
print(f"{len(samples)} samples × {MAX_WORKERS} workers\n")

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    fmap = {executor.submit(process_sample, s): i for i, s in enumerate(samples)}
    for future in tqdm(as_completed(fmap), total=len(fmap), desc=PIPELINE_NAME):
        try:
            r = future.result(); r['idx'] = fmap[future]; results.append(r)
        except Exception as e:
            print(f"  Error on sample {fmap[future]}: {e}")

results.sort(key=lambda x: x['idx'])
predictions = [r['pred']   for r in results]
golds       = [r['gold']   for r in results]
details     = [r['detail'] for r in results]

experiment_end = datetime.now()
total_time = (experiment_end - experiment_start).total_seconds()
print(f"\nDone in {total_time:.1f}s ({total_time/len(samples):.2f}s/sample)")

## Step 11 — Evaluation

In [ ]:
metrics = {'em':0,'f1':0,'sp_em':0,'sp_f1':0,'sp_prec':0,'sp_recall':0,'joint_em':0,'joint_f1':0}
for pred, gold in zip(predictions, golds):
    em = answer_em(pred['answer'], gold['answer'])
    f1 = answer_f1(pred['answer'], gold['answer'])
    sp = sp_metrics(pred['supporting_facts'], gold['supporting_facts'])
    metrics['em'] += em; metrics['f1'] += f1
    metrics['sp_em'] += sp['sp_em']; metrics['sp_f1'] += sp['sp_f1']
    metrics['sp_prec'] += sp['sp_prec']; metrics['sp_recall'] += sp['sp_recall']
    metrics['joint_em'] += em * sp['sp_em']; metrics['joint_f1'] += f1 * sp['sp_f1']
n = len(predictions)
for k in metrics: metrics[k] /= n

print(f"{'='*50}")
print(f"  {PIPELINE_NAME} Results ({n} samples)")
print(f"{'='*50}")
for k, v in metrics.items(): print(f"  {k:20s}: {v:.4f}")
print(f"{'='*50}")

## Step 12 — Save Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
ts = experiment_start.strftime('%Y%m%d_%H%M%S')
out_file = f'{RESULTS_DIR}/single_stage_{ts}_results.json'
with open(out_file, 'w') as f:
    json.dump({'args': {'pipeline': PIPELINE_NAME, 'model': GROQ_MODEL, 'dense_model': DENSE_MODEL,
               'top_k': TOP_K, 'n_samples': N_SAMPLES, 'max_workers': MAX_WORKERS},
               'metrics': metrics, 'timing': {'total_s': total_time, 'avg_s': total_time/n},
               'details': details}, f, indent=2)
print(f"Saved → {out_file}")

## Step 13 — Inspect Predictions & Key Usage

In [ ]:
for d in details[:5]:
    print(f"Q: {d['question']}")
    print(f"  Gold: {d['gold_answer']} | Pred: {d['pred_answer']}")
    print(f"  Retrieved: {[(r['title'], r['sent_id']) for r in d['retrieved_context']]}\n")

print("--- API Key Usage ---")
key_manager.status()